# Fractal Neural Network with Stored Fractal Activation

In [11]:
import numpy as np
import wandb

# Importing from local directory
from src.functions import load_data, preprocess_data, initialize_network, compute_accuracy
from src.optimizers import optimizers
from src.propagation import forward_propagation, backpropagation
from src.alpha_fractal_function import alpha_fractalize, alpha_fractalize_first_derivative, pointwise_fractal
from src.activation import set_fractal_activation

In [12]:
# Define base function f(x) and g(x)

def gauss_RBF(z, c=1):
    return np.exp(-(c * z)**2)

def dgauss_RBF(z, c=1):
    return -2 * (c**2) * z * np.exp(-(c * z)**2)

def ddgauss_RBF(z, c=1):
    return 2 * (c**2) * (2 * (c**2) * (z**2) - 1) * np.exp(-(c * z)**2)

def H5(z, x1=-2, xN=2, c=1):
    z = np.asarray(z, dtype=float)
    dx = xN - x1

    gauss_RBF1   = gauss_RBF(x1, c)
    gauss_RBFN   = gauss_RBF(xN, c)
    gauss_RBF1d  = dgauss_RBF(x1, c)
    gauss_RBFNd  = dgauss_RBF(xN, c)
    gauss_RBF1dd = ddgauss_RBF(x1, c)
    gauss_RBFNdd = ddgauss_RBF(xN, c)

    h1 = (gauss_RBFN - gauss_RBF1 - gauss_RBF1d*dx - 0.5*gauss_RBF1dd*dx**2) / dx**3
    h2 = (3*(gauss_RBF1 - gauss_RBFN) + 2*(gauss_RBF1d + 0.5*gauss_RBFNd)*dx + 0.5*gauss_RBF1dd*dx**2) / dx**4
    h3 = (6*(gauss_RBFN - gauss_RBF1) - 3*(gauss_RBF1d + gauss_RBFNd)*dx + 0.5*(gauss_RBFNdd - gauss_RBF1dd)*dx**2) / dx**5

    dz = z - x1
    return (gauss_RBF1
            + gauss_RBF1d*dz
            + 0.5*gauss_RBF1dd*dz**2
            + h1*dz**3
            + h2*dz**3*(z - xN)
            + h3*dz**3*(z - xN)**2)

def dH5(z, x1=-2, xN=2, c=1):
    z = np.asarray(z, dtype=float)
    dx = xN - x1

    gauss_RBF1   = gauss_RBF(x1, c)
    gauss_RBFN   = gauss_RBF(xN, c)
    gauss_RBF1d  = dgauss_RBF(x1, c)
    gauss_RBFNd  = dgauss_RBF(xN, c)
    gauss_RBF1dd = ddgauss_RBF(x1, c)
    gauss_RBFNdd = ddgauss_RBF(xN, c)

    h1 = (gauss_RBFN - gauss_RBF1 - gauss_RBF1d*dx - 0.5*gauss_RBF1dd*dx**2) / dx**3
    h2 = (3*(gauss_RBF1 - gauss_RBFN) + 2*(gauss_RBF1d + 0.5*gauss_RBFNd)*dx + 0.5*gauss_RBF1dd*dx**2) / dx**4
    h3 = (6*(gauss_RBFN - gauss_RBF1) - 3*(gauss_RBF1d + gauss_RBFNd)*dx + 0.5*(gauss_RBFNdd - gauss_RBF1dd)*dx**2) / dx**5

    dz = z - x1
    dzN = z - xN

    return (gauss_RBF1d
            + gauss_RBF1dd * dz
            + 3 * h1 * dz**2
            + h2 * dz**2 * (3 * dzN + dz)
            + h3 * dz**2 * dzN * (3 * dzN + 2 * dz))

In [13]:
subintervals = 6
alpha = [0.2, 0.1, 0.05, 0.05, 0.1, 0.2]
iterations = 6

# Create and store fractal function and its first derivative in 'store' and 'store_d'
store = alpha_fractalize(gauss_RBF, H5, -2, 2, subintervals, alpha, iterations, dict=True)
store_d = alpha_fractalize_first_derivative(dgauss_RBF, dH5, -2, 2, subintervals, alpha, iterations, dict=True)
set_fractal_activation(store, store_d)

f(-2) = 0.01831563888873418, f(2) = 0.01831563888873418
g(-2) = 0.01831563888873418, g(2) = 0.01831563888873422


In [ ]:
# Training Function taking store and store_d to evaluate activation and derivative via pointwise_fractal()
def train_network(x_train, y_train, x_val, y_val, layer_sizes, optimizer_name, store=None, store_d=None, activation='alpha_fractal', epochs=30, batch_size=32, learning_rate=0.01):
    # If store and store_d are passed directly, register them
    if store is not None and store_d is not None:
        set_fractal_activation(store, store_d)
        activation = (store, store_d)
        
    weights, biases = initialize_network(layer_sizes, init_method="xavier")
    optimizer = optimizers[optimizer_name](learning_rate)
    
    for epoch in range(epochs):
        indices = np.random.permutation(x_train.shape[0])
        x_train, y_train = x_train[indices], y_train[indices]
        
        for i in range(0, x_train.shape[0], batch_size):
            x_batch = x_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]
            
            # forward_propagation and backpropagation use pointwise_fractal(z, store) and pointwise_fractal(z, store_d)
            activations, z_values = forward_propagation(x_batch, weights, biases, activation)
            gradients_w, gradients_b = backpropagation(activations, z_values, weights, y_batch, activation, 0.0)
            optimizer.update(weights, biases, gradients_w, gradients_b)
        
        train_activations, _ = forward_propagation(x_train, weights, biases, activation)
        loss = -np.mean(np.sum(y_train * np.log(train_activations[-1] + 1e-8), axis=1))  # Cross-Entropy Loss

        val_acc = compute_accuracy(x_val, y_val, weights, biases, activation)
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss:.4f} - Val Acc: {val_acc:.4f}")
    
    return weights, biases

# Loading and Preprocessing Fashion-MNIST Dataset
(x_train, y_train), (x_test, y_test) = load_data('Data/fashion_mnist.npz')
x_train, y_train = preprocess_data(x_train, y_train)
x_test, y_test = preprocess_data(x_test, y_test)



Epoch 1/5 - Loss: 0.5218 - Val Acc: 0.8189
Epoch 2/5 - Loss: 0.5330 - Val Acc: 0.8153
Epoch 3/5 - Loss: 0.5823 - Val Acc: 0.8050
Epoch 4/5 - Loss: 0.6801 - Val Acc: 0.7730
Epoch 5/5 - Loss: 0.8609 - Val Acc: 0.7249


In [18]:
# Train Network using stored fractal activation dictionaries (store, store_d)
weights, biases = train_network(
    x_train, y_train, x_test, y_test, 
    layer_sizes=[784, 64, 32, 10], #[784, 128, 64, 10]
    optimizer_name='adam', 
    store=store,
    store_d=store_d,
    epochs=5,
    learning_rate=0.001
)

Epoch 1/5 - Loss: 0.7997 - Val Acc: 0.6977
Epoch 2/5 - Loss: 1.1005 - Val Acc: 0.5800
Epoch 3/5 - Loss: 1.4357 - Val Acc: 0.4422
Epoch 4/5 - Loss: 1.6283 - Val Acc: 0.3646
Epoch 5/5 - Loss: 1.8027 - Val Acc: 0.3022


Epoch 1/5 - Loss: 1.7719 - Val Acc: 0.2377
Epoch 2/5 - Loss: 1.7089 - Val Acc: 0.2536
Epoch 3/5 - Loss: 1.7097 - Val Acc: 0.3106
Epoch 4/5 - Loss: 1.6736 - Val Acc: 0.2615
Epoch 5/5 - Loss: 1.5618 - Val Acc: 0.3001

In [15]:
# Evaluate Model Accuracy on Test Dataset
test_acc = compute_accuracy(x_test, y_test, weights, biases, activation='alpha_fractal')
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Test Accuracy: 72.49%
